In [9]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')
import joblib
import os
import shap
from xgboost import XGBRegressor
from sklearn.model_selection import (train_test_split, KFold, cross_val_score, GridSearchCV,learning_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn .metrics import (mean_absolute_error, mean_squared_error, r2_score)
os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

df = pd.read_csv(r"C:\Users\niluc\Downloads\PROJECT\Agriculture\Data\agri_clean.csv")
X = df.drop ('yield_tonnes_ha', axis=1)
y = df['yield_tonnes_ha']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
cv = KFold(n_splits=5, shuffle=True, random_state=42)


models = {
    'Linear Regression':  LinearRegression(),
    'Random Forest':   RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBOOST':       XGBRegressor(n_estimators=100, max_depth=3),
}
results={}
for name, m in models.items():
    m.fit(X_train_s, y_train)
    pred = m.predict(X_test_s)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    cv_r2 = cross_val_score(m, X_train_s,y_train, cv=cv, scoring='r2')
    results[name] = {'mae': mae, 'rmse': rmse,'r2': r2, 'cv_r2': cv_r2, 'model': m, 'pred': pred}
    print(f'\n==={name}===')
    print(f'MAE:  {mae:.4f} t/ha (avg error = {mae*1000:.0f} kg/ha')
    print(f'RMSE: {rmse:.4f} t/ha')
    print(f'R2:   {r2:.4f} ({r2*100:.1f}% variance explained)')
    print(f'CV R2: {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}')

    #GridSearchCV
param_grid = {
    'n_estimators': [300, 500],
    'max_depth': [4, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha':[0.1,1.0],
     'reg_lambda': [0.5, 1.0]
    
}
grid = GridSearchCV(
    XGBRegressor(random_state = 42),
    param_grid,
    scoring='r2',
    n_jobs=-1,
    verbose=2
    )
grid.fit(X_train_s, y_train)
best_model = grid.best_estimator_
print(f'\nBest params: {grid.best_params_}')
print(f'n\Best CV R2: {grid.best_score_:.4f}')

pred_best = best_model.predict(X_test_s)
mae_best = mean_absolute_error(y_test, pred_best)
r2_best = r2_score(y_test, pred_best)
print(f'Tuned model - MAE: {mae_best:.4f} | R2: {r2_best:.4f}')

#Actual vs Predicted
plt.figure (figsize =(7,5))
plt.scatter(y_test, pred_best, alpha=0.3, s=10, color='#1B8CA6')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Yield (t/ha)')
plt.ylabel('Predicted Yields (t/ha)')
plt.title('Residual plot - check for bias')
plt.legend();
plt.tight_layout()
plt.savefig('reports/actual_vs_predicted.png', dpi=120)

#learning curves
train_sizes,train_sc,val_sc = learning_curve(
    best_model, X_train_s, y_train, cv=cv,scoring='r2', train_sizes=np.linspace(0.1,1.0,10), n_jobs=-1,random_state=42)
plt.figure(figsize=(8,5))
plt.plot(train_sizes, train_sc.mean(axis=1), 'o-', color='#1B8CA6', label='TRAIN R2')
plt.plot(train_sizes, val_sc.mean(axis=1), 'o-',color='#C0392B', label='VAL R2')
plt.xlabel('Training Size');
plt.ylabel('R2 Score')
plt.title('Learning curves');
plt.legend();
plt.tight_layout()
plt.savefig('reports/Learning Curves.png', dpi=120)
print('Done')

X_test_df = pd.DataFrame(X_test_s, columns=X.columns).sample(200, random_state=42)
explainer = shap.TreeExplainer(best_model)
shap_vals = explainer.shap_values(X_test_df)
plt.figure(figsize=(10,8))
shap.summary_plot(shap_vals, X_test_df, plot_type ='bar', show=False)
plt.title('SHAP Feature Importances - Yield Drivers')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=120, bbox_inches ='tight')
plt.close()

plt.figure(figsize=(10,10))
shap.summary_plot(shap_vals, X_test_df,plot_type="dot",show=False)
plt.title('SHAP summary plot - How features affect yield')
plt.tight_layout()
plt.savefig('shap_importance_dot.png',dpi=120)
plt.close()
print('SHAP saved.')
joblib.dump(best_model, 'models/agri_model.pkl')
joblib.dump(scaler, 'models/agri_scaler.pkl')
joblib.dump(list(X.columns), 'models/feature_names.pkl')
print('Model saved.')
    




===Linear Regression===
MAE:  0.7425 t/ha (avg error = 743 kg/ha
RMSE: 1.1729 t/ha
R2:   0.6479 (64.8% variance explained)
CV R2: 0.6244 +/- 0.0164

===Random Forest===
MAE:  0.4876 t/ha (avg error = 488 kg/ha
RMSE: 0.9896 t/ha
R2:   0.7494 (74.9% variance explained)
CV R2: 0.7842 +/- 0.0303

===XGBOOST===
MAE:  0.4858 t/ha (avg error = 486 kg/ha
RMSE: 0.8914 t/ha
R2:   0.7966 (79.7% variance explained)
CV R2: 0.7979 +/- 0.0272
Fitting 5 folds for each of 128 candidates, totalling 640 fits

Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300, 'reg_alpha': 1.0, 'reg_lambda': 0.5, 'subsample': 1.0}
n\Best CV R2: 0.8181
Tuned model - MAE: 0.4474 | R2: 0.7977
Done
SHAP saved.
Model saved.
